Steps:
Load the Dataset: Load the CSV file that we just saved.

Preprocessing: Ensure that the features (CGPA and Hours Spent) and the target (Domain) are correctly handled.

Train the Model: Use RandomForestClassifier to train the model on the dataset.

Make Predictions: Use the trained model to predict the domain for a given user input (CGPA and hours spent).

In [1]:
import pandas as pd
import numpy as np
import random
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import joblib

# Load the dataset from the previously saved CSV file
df = pd.read_csv("balanced_domain_recommendation_data.csv")

# Encode categorical target variable ('Domain')
label_encoder = LabelEncoder()
df["Domain"] = label_encoder.fit_transform(df["Domain"])

# Define features (X) and target (y)
X = df[["CGPA", "Hours Spent"]]  # Features: CGPA, Hours Spent
y = df["Domain"]  # Target: Domain

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Classifier
model = RandomForestClassifier(n_estimators=500, max_depth=20, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

# Save the model and label encoder for future use
joblib.dump(model, "domain_recommendation_model.pkl")
joblib.dump(label_encoder, "domain_label_encoder.pkl")
print("✅ Model and Label Encoder saved.")

# Function to predict domain for a new user
def recommend_domain(user_input):
    """
    user_input: Dictionary containing user data (CGPA, Hours Spent, etc.)
    Example:
    {
        'CGPA': <value>, 'Study Hours': <value>
    }
    """
    # Validate user input
    if not isinstance(user_input["CGPA"], (int, float)) or not isinstance(user_input["Study Hours"], (int, float)):
        return "❌ Invalid data type for CGPA or Study Hours. Please enter numeric values."

    # Prepare input data for prediction
    input_data = np.array([[user_input["CGPA"], user_input["Study Hours"]]])

    # Predict the domain (encoded)
    predicted_domain_encoded = model.predict(input_data)[0]

    # Decode the predicted domain using the label encoder
    predicted_domain = label_encoder.inverse_transform([predicted_domain_encoded])[0]
    return f"Based on your CGPA and study hours, we recommend: {predicted_domain}"

# Example user input (CGPA: 6.5, Study Hours: 4)
user_input = {"CGPA": 6.5, "Study Hours": 4}
domain_recommendation = recommend_domain(user_input)
print(domain_recommendation)


Model Accuracy: 0.79
✅ Model and Label Encoder saved.
Based on your CGPA and study hours, we recommend: Web Development


C:\Users\Muqeem\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Expected Outcome: 

Improved Accuracy: The model's accuracy should improve after applying hyperparameter tuning.

Best Hyperparameters: The best hyperparameters found by GridSearchCV will be printed, and the tuned model will be saved.

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib
import os

# Load the dataset from the previously saved CSV file
df = pd.read_csv("balanced_domain_recommendation_data.csv")

# Encode categorical target variable ('Domain')
label_encoder = LabelEncoder()
df["Domain"] = label_encoder.fit_transform(df["Domain"])

# Define features (X) and target (y)
X = df[["CGPA", "Hours Spent"]]  # Features: CGPA, Hours Spent
y = df["Domain"]  # Target: Domain

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the model
rf_model = RandomForestClassifier(random_state=42)

# Define hyperparameters grid for GridSearchCV
param_grid = {
    "n_estimators": [100, 200, 300, 500, 1000],  # Number of trees
    "max_depth": [10, 20, 30, 50, None],  # Max depth of the tree
    "min_samples_split": [2, 5, 10, 20],  # Min samples required to split a node
    "min_samples_leaf": [1, 2, 5, 10],  # Min samples required at a leaf node
    "max_features": ["sqrt", "log2", None],  # Max features to consider for splits
    "bootstrap": [True, False],  # Whether to use bootstrap sampling
    "criterion": ["gini", "entropy"]  # Criterion for measuring the quality of a split
}

# Perform GridSearchCV to find the best parameters
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)

# Fit the model
grid_search.fit(X_train, y_train)

# Get the best model after tuning
best_rf_model = grid_search.best_estimator_

# Print the best hyperparameters found
print("Best Hyperparameters from GridSearchCV:", grid_search.best_params_)

# Evaluate the tuned model
y_pred = best_rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Optimized Random Forest Model Accuracy: {accuracy:.2f}")

# Define file path for saving the model and label encoder together
model_and_encoder_file_path = "C:/Users/Muqeem/Documents/domain_recommendation_model_and_encoder.pkl"

# Check if the directory is writable
if os.access(os.path.dirname(model_and_encoder_file_path), os.W_OK):
    # Save both model and label encoder as a dictionary
    model_and_encoder = {
        "model": best_rf_model,
        "label_encoder": label_encoder
    }
    joblib.dump(model_and_encoder, model_and_encoder_file_path)
    print(f"✅ Model and Label Encoder saved as '{model_and_encoder_file_path}'")
else:
    print(f"❌ Error: No write permissions to save the model and label encoder to '{model_and_encoder_file_path}'")


Fitting 3 folds for each of 4800 candidates, totalling 14400 fits
Best Hyperparameters from GridSearchCV: {'bootstrap': False, 'criterion': 'entropy', 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100}
Optimized Random Forest Model Accuracy: 0.78
✅ Model and Label Encoder saved as 'C:/Users/Muqeem/Documents/domain_recommendation_model_and_encoder.pkl'


Code to Predict Domain for New Users:

In [3]:
import joblib
import numpy as np

# Load the saved model and label encoder
model_and_encoder = joblib.load("C:/Users/Muqeem/Documents/domain_recommendation_model_and_encoder.pkl")

# Extract model and label encoder
domain_model = model_and_encoder["model"]
domain_label_encoder = model_and_encoder["label_encoder"]

def recommend_domain(user_input):
    """
    user_input: Dictionary containing user data (CGPA, Hours Spent, etc.)
    Example:
    {
        'CGPA': <value>, 'Study Hours': <value>
    }
    """
    # Validate user input
    if not isinstance(user_input["CGPA"], (int, float)) or not isinstance(user_input["Study Hours"], (int, float)):
        return "❌ Invalid data type for CGPA or Study Hours. Please enter numeric values."

    # Prepare input data for prediction
    input_data = np.array([[user_input["CGPA"], user_input["Study Hours"]]])

    # Predict the domain (encoded)
    predicted_domain_encoded = domain_model.predict(input_data)[0]

    # Decode the predicted domain using the label encoder
    predicted_domain = domain_label_encoder.inverse_transform([predicted_domain_encoded])[0]
    return f"Based on your CGPA and study hours, we recommend: {predicted_domain}"

# Example user input (CGPA: 6.5, Study Hours: 4)
user_input = {"CGPA": 6.5, "Study Hours": 4}
domain_recommendation = recommend_domain(user_input)
print(domain_recommendation)


Based on your CGPA and study hours, we recommend: Web Development


C:\Users\Muqeem\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
